In [1]:
# Import libraries and AIF360
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing

# number of random train/test splits
N_RUNS = 10                                
privileged = [{"sensitive": 1}]
unprivileged = [{"sensitive": 0}]

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


In [2]:
# Load/clean Adult dataset
def load_adult():
    df = pd.read_csv("adult.csv").drop(columns=["fnlwgt", "education"])
    df["label"] = (df["income"] == ">50K").astype(int)
    df["sensitive"] = (df["sex"] == "Male").astype(int)
    return df.drop(columns=["income", "sex"])
    
# Load/clean German dataset
def load_german():
    df = pd.read_csv("german.data", sep=" ", header=None, names=[f"col{i}" for i in range(21)])
    df["label"] = (df["col20"] == 1).astype(int)
    df["sensitive"] = df["col8"].isin(["A91", "A93", "A94"]).astype(int)
    return df.drop(columns=["col20", "col8"])
    
# Load/clean COMPAS dataset
def load_compas():
    df = pd.read_csv("propublica_data_for_fairml.csv")
    others = ["Asian", "Hispanic", "Native_American", "Other"]
    black = df["African_American"] == 1
    white = (df["African_American"] == 0) & (df[others].sum(axis=1) == 0)
    df = df[black | white].copy()
    df["label"] = 1 - df["Two_yr_Recidivism"]
    df["sensitive"] = (df["African_American"] == 0).astype(int)
    return df.drop(columns=["Two_yr_Recidivism", "African_American", "score_factor"] + others)

In [3]:
# scale all features between 0 and 1
def prepare(df):
    df = pd.get_dummies(df).astype(float)
    features = [c for c in df.columns if c != "label"]
    df[features] = MinMaxScaler().fit_transform(df[features])
    return df    
# Load/preprocess the datasets
datasets = {"Adult": prepare(load_adult()),
            "German": prepare(load_german()),
            "COMPAS": prepare(load_compas())}

In [4]:
# Convert dataframe to the AIF360 format
def to_aif(df):
    return BinaryLabelDataset(df=df, label_names=["label"], protected_attribute_names=["sensitive"],
                              favorable_label=1, unfavorable_label=0)

# Compute DP_Diff, DIR, and consistency for the AIF360 dataset
def fairness(ds):
    m = BinaryLabelDatasetMetric(ds, unprivileged_groups=unprivileged, privileged_groups=privileged)
    return {"DP_Diff": m.statistical_parity_difference(),
            "DIR": m.disparate_impact(),
            "Consistency": m.consistency()[0]}

In [5]:
# Calculate baseline fairness metrics before splitting
before = {name: fairness(to_aif(df)) for name, df in datasets.items()}
pd.DataFrame(before).T.round(3)

,DP_Diff,DIR,Consistency
Adult,-0.196,0.358,0.833
German,-0.075,0.897,0.747
COMPAS,-0.132,0.783,0.605


In [6]:
# Run 10 train/test splits per dataset, and comparing against baseline
results = []
for name, df in datasets.items():
    features = [c for c in df.columns if c not in ("label", "sensitive")]
    groups = df["label"].astype(int).astype(str) + df["sensitive"].astype(int).astype(str)

    for seed in range(N_RUNS):
        train, test = train_test_split(df, test_size=0.3, stratify=groups, random_state=seed)
        train_ds, test_ds = to_aif(train), to_aif(test)

        # Only looks at training data
        rw = Reweighing(unprivileged_groups=unprivileged, privileged_groups=privileged).fit(train_ds)
        weights = rw.transform(train_ds).instance_weights

        # classifier without weights and with the weights
        for approach, w in [("Baseline", None), ("Reweighing", weights)]:
            model = LogisticRegression(max_iter=1000).fit(train[features], train["label"], sample_weight=w)
            predictions = model.predict(test[features])

            predicted_ds = test_ds.copy()
            predicted_ds.labels = predictions.reshape(-1, 1)

            row = fairness(predicted_ds)
            row["Accuracy"] = accuracy_score(test["label"], predictions)
            row["Balanced_acc"] = balanced_accuracy_score(test["label"], predictions)
            row["Dataset"], row["Approach"] = name, approach
            results.append(row)
    print("done", name)

# Convert into dataframe
results = pd.DataFrame(results)

done Adult
done German
done COMPAS


In [7]:
# Compute the mean fairness and accuracy for the 10 splits
results.groupby(["Dataset", "Approach"], sort=False).mean().round(3)

DP_Diff    DIR  Consistency  Accuracy  Balanced_acc
Dataset Approach                                                       
Adult   Baseline     -0.177  0.300        0.952     0.849         0.760
        Reweighing   -0.088  0.583        0.952     0.843         0.743
German  Baseline     -0.065  0.918        0.827     0.751         0.669
        Reweighing   -0.056  0.930        0.827     0.747         0.662
COMPAS  Baseline     -0.246  0.664        0.991     0.665         0.661
        Reweighing   -0.246  0.667        0.990     0.665         0.661

In [8]:
# Compute standard deviation for the 10 splits
results.groupby(["Dataset", "Approach"], sort=False).std().round(3)

DP_Diff    DIR  Consistency  Accuracy  Balanced_acc
Dataset Approach                                                       
Adult   Baseline      0.007  0.024        0.001     0.003         0.004
        Reweighing    0.005  0.022        0.002     0.003         0.005
German  Baseline      0.060  0.074        0.019     0.019         0.028
        Reweighing    0.063  0.077        0.016     0.017         0.026
COMPAS  Baseline      0.017  0.024        0.001     0.008         0.007
        Reweighing    0.016  0.023        0.001     0.007         0.007